# PAP - Fine-tuning a BERT based model

### Load and preprocess data

In [1]:
from utils import load_PAP_datasets, data_loaders, training_loop
from transformers import BertTokenizer, DataCollatorWithPadding

# Load raw data
raw_datasets = load_PAP_datasets(datasets_path='data/pap/train-dev-test-split/binary')

# Init pre-trained tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Simple tokenize function
def tokenize_function(dataset, truncation=True):
    return tokenizer(dataset['text'])

# Tokenize raw data
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Clear datasets dictionary
tokenized_datasets = tokenized_datasets.remove_columns(['text', 'original_label'])
tokenized_datasets = tokenized_datasets.rename_column('label', 'labels')
tokenized_datasets.set_format('torch')

# Initialize the data collator with padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# DEBUG
print('raw datasets:\n', raw_datasets, '\n')
print('tokenized datasets:\n', tokenized_datasets)

# Initialize DataLoader for train, validation and test splits
dataloaders = data_loaders(tokenized_datasets, data_collator)
train_dataloader = dataloaders['train']
eval_dataloader = dataloaders['validation']
test_dataloader = dataloaders['test']


/Users/saponaro/Developer/uni/noble-owl/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 216/216 [00:00<00:00, 10577.33 examples/s]


raw datasets:
 DatasetDict({
    train: Dataset({
        features: ['text', 'original_label', 'label'],
        num_rows: 1728
    })
    validation: Dataset({
        features: ['text', 'original_label', 'label'],
        num_rows: 216
    })
    test: Dataset({
        features: ['text', 'original_label', 'label'],
        num_rows: 216
    })
}) 

tokenized datasets:
 DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1728
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 216
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 216
    })
})


Initialize DataLoaders for __train__ and __validation__ splits.

## Training loop

In [2]:
from transformers import BertForSequenceClassification

# Initialize a pre-trained BERT model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Start training
training_loop(train_dataloader, model)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 27/27 [01:29<00:00,  3.31s/it]


## Evaluation

In [24]:
import torch
import evaluate

# Run on GPU available
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# Define evaluation metrics
metrics = {
    'accuracy': evaluate.load('accuracy'),
    'precision': evaluate.load('precision'),
    'recall': evaluate.load('recall'),
    'f1': evaluate.load('f1'),
    #'roc_auc': evaluate.load('roc_auc'),
}

# Load ROC AUC matric
roc_auc = evaluate.load('roc_auc')

# Set model in evaluation mode
model.eval()

for batch in eval_dataloader:
    
    # Move batch data to the specified device (GPU or CPU)
    batch = {k: v.to(device) for k, v in batch.items()}
    
    # Forward pass
    with torch.no_grad():
        outputs = model(**batch)

    # Extract logits and predictions
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

    # Apply softmax to convert logits to probabilities
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    
    # Extract probabilities for the positive class
    positive_probabilities = probabilities[:, 1].to(device).numpy()

    # Update metrics for accuracy, precision, recall, and F1
    for metric in metrics.values():
        metric.add_batch(predictions=predictions, references=batch['labels'])

    # Update ROC AUC metric
    roc_auc.add_batch(prediction_scores=positive_probabilities, references=batch['labels'])

# Compute metrics for accuracy, precision, recall, and F1
eval_dict = {}
for metric in metrics.values():
    eval_dict.update(metric.compute())

# Compute ROC AUC metric
eval_dict.update(roc_auc.compute())



eval_dict

{'accuracy': 0.7129629629629629,
 'precision': 0.7129629629629629,
 'recall': 1.0,
 'f1': 0.8324324324324325,
 'roc_auc': 0.7561793045664014}

prediction on test data

In [25]:
# Run on GPU available
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# Define evaluation metrics
metrics = {
    'accuracy': evaluate.load('accuracy'),
    'precision': evaluate.load('precision'),
    'recall': evaluate.load('recall'),
    'f1': evaluate.load('f1'),
    #'roc_auc': evaluate.load('roc_auc'),
}

# Load ROC AUC matric
roc_auc = evaluate.load('roc_auc')

# Set model in evaluation mode
model.eval()

for batch in test_dataloader:
    
    # Move batch data to the specified device (GPU or CPU)
    batch = {k: v.to(device) for k, v in batch.items()}
    
    # Forward pass
    with torch.no_grad():
        outputs = model(**batch)

    # Extract logits and predictions
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

    # Apply softmax to convert logits to probabilities
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    
    # Extract probabilities for the positive class
    positive_probabilities = probabilities[:, 1].to(device).numpy()

    # Update metrics for accuracy, precision, recall, and F1
    for metric in metrics.values():
        metric.add_batch(predictions=predictions, references=batch['labels'])

    # Update ROC AUC metric
    roc_auc.add_batch(prediction_scores=positive_probabilities, references=batch['labels'])

# Compute metrics for accuracy, precision, recall, and F1
eval_dict = {}
for metric in metrics.values():
    eval_dict.update(metric.compute())

# Compute ROC AUC metric
eval_dict.update(roc_auc.compute())